# 01 · Perfil socioeconómico de hogares por AGEB (ciudad según `src/config.py`)

**Salida:** una fila por AGEB urbana con Latitud/Longitud y, para cada categoría, `HHs`, `%` e `Indice`:

| Grupo | Categorías |
|---|---|
| Tamaño del hogar | 2 o menos, 3, 4, 5, 6+ personas |
| Niños (edad del **menor** de 18 en el hogar) | < 6, 6–11, 12–17, sin niños |
| Edad del jefe(a) de hogar | < 35, 35–44, 45–54, 55–64, 65+ |
| NSE (Regla **AMAI 2024**, vigente; mismos puntos y cortes que la 2022) | A/B, C+, C, C-, D+, D/E |

**Fuentes oficiales** (todas las URLs viven en `src/config.py` → `FUENTES`; la celda 1 las muestra y
las verifica con HEAD al correr). Verificadas el 2026-09-24:

| # | Fuente | Edición | Página oficial | Uso |
|---|---|---|---|---|
| 1 | INEGI · Censo 2020, Principales resultados por AGEB y manzana urbana | 2020 | https://www.inegi.org.mx/programas/ccpv/2020/#datos_abiertos | Totales por AGEB (restricciones) y hogares por manzana |
| 1b | INEGI · Censo 2020, Principales resultados por localidad (ITER) | 2020 | https://www.inegi.org.mx/programas/ccpv/2020/#datos_abiertos | Hogares de localidades rurales (fuera de las AGEB urbanas) |
| 1c | INEGI · Encuesta Intercensal 2025, resultados por municipio (publicada 2026-09-22) | 2025 | https://www.inegi.org.mx/programas/eic/2025/ | Crecimiento de población 2020→2025 por municipio (actualiza la demanda del notebook 04) |
| 2 | INEGI · Censo 2020, Microdatos del cuestionario ampliado | 2020 | https://www.inegi.org.mx/programas/ccpv/2020/#microdatos | Hogares donantes |
| 3 | INEGI · Marco Geoestadístico de la Encuesta Intercensal 2025 (el más reciente) | edición 2026, datos a nov-2025 (UPC 794551196649; `MG_VERSION` en config) | https://www.inegi.org.mx/app/biblioteca/ficha.html?upc=794551196649 | Polígonos de AGEB, manzana y localidad |
| 4 | INEGI · ENIGH 2024, nueva serie (la más reciente) | 2024 | https://www.inegi.org.mx/programas/enigh/nc/2024/#datos_abiertos | Imputar *número* de baños completos y autos |
| 5 | AMAI · Regla NSE 2024 | vigente desde ene-2024 | https://www.amai.org/NSE/index.php?queVeo=NSE2024 | Puntos y cortes del NSE (`src/nse.py`) |

URL de descarga por estado (`{ENT}` = clave INEGI, `{abrev}` y `{slug}` en `config.py`):
1. `https://www.inegi.org.mx/contenidos/programas/ccpv/2020/datosabiertos/ageb_manzana/ageb_mza_urbana_{ENT}_cpv2020_csv.zip`
1b. `https://www.inegi.org.mx/contenidos/programas/ccpv/2020/datosabiertos/iter/iter_{ENT}_cpv2020_csv.zip`
2. `https://www.inegi.org.mx/contenidos/programas/ccpv/2020/microdatos/Censo2020_CA_{abrev}_csv.zip`
3. `https://www.inegi.org.mx/contenidos/productos/prod_serv/contenidos/espanol/bvinegi/productos/geografia/marcogeo/794551196649/{ENT}_{slug}.zip` (antes 794551163061 = MG 2025)
4. `https://www.inegi.org.mx/contenidos/programas/enigh/nc/2024/datosabiertos/conjunto_de_datos_enigh2024_ns_csv.zip`
5. Nota metodológica: `https://www.amai.org/descargas/NOTA_METODOLOGICA_NSE_AMAI_2024_v6.pdf`

**Por qué Censo 2020 y no la Encuesta Intercensal 2025:** la EIC 2025 (publicada el 2026-09-22) es una muestra
representativa por estado, municipio y localidades de 50 mil+ habitantes; **no publica resultados por AGEB ni
manzana**, así que el Censo 2020 sigue siendo la fuente más reciente a ese nivel.

**Marco Geoestadístico 2025 con datos del Censo 2020:** las claves de AGEB coinciden igual que con el MG 2020;
unas pocas manzanas del Censo ya no existen en el MG 2025 (se reporta cuántas en la sección 8).

**Por qué hace falta modelar:** el Censo por AGEB no publica tamaño de hogar, edad del jefe ni NSE.
Esas variables sí existen en la muestra censal, pero solo con desagregación municipal / localidad.
Se usa **microsimulación espacial (IPU)**: para cada AGEB se reponderan los hogares de la muestra
hasta reproducir exactamente lo que el Censo publica para esa AGEB (hogares, autos, internet,
computadora, dormitorios, personas por edad, ocupados, escolaridad…). Con esos pesos se estiman las
distribuciones anteriores.

**Indice** = % de la AGEB / % de la referencia × 100 (referencia configurable en `src/config.py`:
total de la ZM o total del estado).

In [1]:
import sys, zipfile
from pathlib import Path

BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())   # raíz del repo
sys.path.insert(0, str(BASE / "src"))
import numpy as np
import pandas as pd
import geopandas as gpd

import config as C
import nse
from descargas import descargar_fuente, extraer, registrar_fuentes, verificar_fuentes
from IPython.display import display
from microsim import ipu

pd.set_option("display.width", 200, "display.max_columns", 40)


# reproducibilidad: ciudad, versiones y semilla (no hay pasos aleatorios en este notebook)
import platform
print(f"Repo: {BASE} | ciudad: {C.ZM_NOMBRE} (CIUDAD={C.CIUDAD}) | Python {platform.python_version()} · pandas {pd.__version__} · numpy {np.__version__}")

Repo: C:\Users\KIN\Desktop\Kin\mx-retail-geodemographics | ciudad: Estado de Hidalgo (QA) (CIUDAD=qasur_13) | Python 3.14.6 · pandas 3.0.5 · numpy 2.5.1


## 1. Fuentes: de dónde se descarga cada archivo (verificación en vivo) y descarga
Descarga con reanudación: los servidores de INEGI suelen cortar la conexión.

In [2]:
print("Regla NSE:", C.REGLA_NSE["nombre"], "|", C.REGLA_NSE["pagina"])
print(f"Ciudad: {C.ZM_NOMBRE} · {C.NOM_ENT} ({C.ENT}) · municipios: " + ", ".join(f"{v} ({k})" for k, v in C.ZM_MUNICIPIOS.items()))
display(verificar_fuentes(C.FUENTES, ["ageb", "iter", "micro", "mg", "enigh", "eic"]))

Regla NSE: Regla NSE AMAI 2024 (vigente desde enero 2024; mismos puntos y cortes que la Regla 2022) | https://www.amai.org/NSE/index.php?queVeo=NSE2024
Ciudad: Estado de Hidalgo (QA) · Hidalgo (13) · municipios: Acatlán (001), Acaxochitlán (002), Actopan (003), Agua Blanca de Iturbide (004), Ajacuba (005), Alfajayucan (006), Almoloya (007), Apan (008), El Arenal (009), Atitalaquia (010), Atlapexco (011), Atotonilco el Grande (012), Atotonilco de Tula (013), Calnali (014), Cardonal (015), Cuautepec de Hinojosa (016), Chapantongo (017), Chapulhuacán (018), Chilcuautla (019), Eloxochitlán (020), Emiliano Zapata (021), Epazoyucan (022), Francisco I. Madero (023), Huasca de Ocampo (024), Huautla (025), Huazalingo (026), Huehuetla (027), Huejutla de Reyes (028), Huichapan (029), Ixmiquilpan (030), Jacala de Ledezma (031), Jaltocán (032), Juárez Hidalgo (033), Lolotla (034), Metepec (035), San Agustín Metzquititlán (036), Metztitlán (037), Mineral del Chico (038), Mineral del Monte (039), La 

,ciudad,cobertura,nombre,edición,estado,MB,última modificación (servidor),página oficial,url de descarga
fuente,,,,,,,,,
ageb,Estado de Hidalgo (QA),Hidalgo (13) → se filtran los 84 municipios de...,Censo de Población y Vivienda 2020 · Principal...,"2020 (datos abiertos, CSV)",OK,5.9,"Mon, 26 Jul 2021 15:15:45 GMT",https://www.inegi.org.mx/programas/ccpv/2020/#...,https://www.inegi.org.mx/contenidos/programas/...
iter,Estado de Hidalgo (QA),Hidalgo (13) → se filtran los 84 municipios de...,Censo de Población y Vivienda 2020 · Principal...,"2020 (datos abiertos, CSV)",OK,1.4,"Thu, 19 May 2022 21:00:28 GMT",https://www.inegi.org.mx/programas/ccpv/2020/#...,https://www.inegi.org.mx/contenidos/programas/...
micro,Estado de Hidalgo (QA),Hidalgo (13) → se filtran los 84 municipios de...,Censo de Población y Vivienda 2020 · Microdato...,2020 (CSV por entidad),OK,17.2,"Mon, 26 Jul 2021 15:05:32 GMT",https://www.inegi.org.mx/programas/ccpv/2020/#...,https://www.inegi.org.mx/contenidos/programas/...
mg,Estado de Hidalgo (QA),Hidalgo (13) → se filtran los 84 municipios de...,Marco Geoestadístico Encuesta Intercensal 2025...,Encuesta Intercensal 2025 (edición 2026) (UPC ...,OK,109.6,"Tue, 22 Sep 2026 18:14:26 GMT",https://www.inegi.org.mx/app/biblioteca/ficha....,https://www.inegi.org.mx/contenidos/productos/...
enigh,Estado de Hidalgo (QA),Nacional → se usan hogares urbanos de Hidalgo ...,Encuesta Nacional de Ingresos y Gastos de los ...,2024 (publicada 2025-07-30; la más reciente),OK,103.1,"Wed, 12 Nov 2025 22:16:56 GMT",https://www.inegi.org.mx/programas/enigh/nc/20...,https://www.inegi.org.mx/contenidos/programas/...
eic,Estado de Hidalgo (QA),Nacional → hogares y viviendas por municipio d...,Encuesta Intercensal 2025 · Principales result...,2025 (publicada 2026-09-22; estimaciones con CV),OK,8.3,"Tue, 22 Sep 2026 10:44:02 GMT",https://www.inegi.org.mx/programas/eic/2025/,https://www.inegi.org.mx/contenidos/programas/...


In [3]:
for k in ["ageb", "iter", "micro", "mg", "enigh", "eic"]:
    print(f"{k}: {C.URLS[k]}")
    descargar_fuente(C.FUENTES[k])



D_AGEB = extraer(C.ARCHIVOS["ageb"])
D_ITER = extraer(C.ARCHIVOS["iter"])
D_MICRO = extraer(C.ARCHIVOS["micro"])
D_MG = extraer(C.ARCHIVOS["mg"])
D_ENIGH = extraer(C.ARCHIVOS["enigh"])
D_EIC = extraer(C.ARCHIVOS["eic"])

# registro de las versiones exactas usadas (para depurar / reproducir): data/processed/<ciudad>/fuentes_usadas_01.csv
registrar_fuentes(C.FUENTES, ["ageb", "iter", "micro", "mg", "enigh", "eic"], C.PROC / "fuentes_usadas_01.csv")[["fuente", "edicion", "archivo_local", "bytes", "descargado"]]

ageb: https://www.inegi.org.mx/contenidos/programas/ccpv/2020/datosabiertos/ageb_manzana/ageb_mza_urbana_13_cpv2020_csv.zip
[ok] ya existe ageb_mza_urbana_13_cpv2020_csv.zip
iter: https://www.inegi.org.mx/contenidos/programas/ccpv/2020/datosabiertos/iter/iter_13_cpv2020_csv.zip
[ok] ya existe iter_13_cpv2020_csv.zip
micro: https://www.inegi.org.mx/contenidos/programas/ccpv/2020/microdatos/Censo2020_CA_hgo_csv.zip


[ok] ya existe Censo2020_CA_hgo_csv.zip
mg: https://www.inegi.org.mx/contenidos/productos/prod_serv/contenidos/espanol/bvinegi/productos/geografia/marcogeo/794551196649/13_hidalgo.zip


[ok] ya existe mg2025eic_13_hidalgo.zip
enigh: https://www.inegi.org.mx/contenidos/programas/enigh/nc/2024/datosabiertos/conjunto_de_datos_enigh2024_ns_csv.zip


[ok] ya existe enigh2024_ns_csv.zip
eic: https://www.inegi.org.mx/contenidos/programas/eic/2025/datosabiertos/conjunto_de_datos_eic2025_105_csv.zip


[ok] ya existe conjunto_de_datos_eic2025_105_csv.zip


,fuente,edicion,archivo_local,bytes,descargado
0,ageb,"2020 (datos abiertos, CSV)",C:\Users\KIN\Desktop\Kin\mx-retail-geodemograp...,5878783,2026-10-01T19:51:04
1,iter,"2020 (datos abiertos, CSV)",C:\Users\KIN\Desktop\Kin\mx-retail-geodemograp...,1393263,2026-10-01T19:38:08
2,micro,2020 (CSV por entidad),C:\Users\KIN\Desktop\Kin\mx-retail-geodemograp...,17225011,2026-10-01T19:51:24
3,mg,Encuesta Intercensal 2025 (edición 2026) (UPC ...,C:\Users\KIN\Desktop\Kin\mx-retail-geodemograp...,109588420,2026-10-01T19:52:55
4,enigh,2024 (publicada 2025-07-30; la más reciente),C:\Users\KIN\Desktop\Kin\mx-retail-geodemograp...,103139139,2026-09-29T09:56:26
5,eic,2025 (publicada 2026-09-22; estimaciones con CV),C:\Users\KIN\Desktop\Kin\mx-retail-geodemograp...,8292782,2026-09-29T09:56:41


## 2. Imputación de baños completos y autos con ENIGH 2024

El Censo pregunta **si** hay auto y **si** hay regadera + excusado, pero no cuántos. La regla AMAI
da distintos puntos para 1 vs 2+. Se entrena un logit ponderado en hogares urbanos de la península
(`C.ENIGH_ESTADOS`: el estado y sus vecinos) con variables que existen en ambas fuentes.

**Validación:** en la propia ENIGH se compara el NSE con conteos reales vs el NSE obtenido tras
"censalizar" la información (convertir a sí/no e imputar).

In [4]:
enigh = nse.cargar_enigh(D_ENIGH, C.ENIGH_ESTADOS)
modelos = nse.entrenar_modelos(enigh)

real = nse.nse_real_enigh(enigh)
censal = enigh.assign(banos1=(enigh.n_banos >= 1).astype(int), auto1=(enigh.n_autos >= 1).astype(int),
                      ocup_model=enigh.ocup)
prob_val = nse.distribucion_nse(censal, modelos)

yuc = enigh.ent == C.ENT
val = pd.DataFrame({
    f"ENIGH real ({C.NOM_ENT} urbano)": pd.Series(enigh.factor[yuc].values, index=real[yuc]).groupby(level=0).sum(),
    f"Imputado ({C.NOM_ENT} urbano)": (prob_val[yuc].mul(enigh.factor[yuc], axis=0)).sum(),
}).reindex(nse.NIVELES)
val = (val / val.sum() * 100).round(1)
val["dif pp"] = (val.iloc[:, 1] - val.iloc[:, 0]).round(1)
print(f"Hogares ENIGH usados: {len(enigh):,} ({C.NOM_ENT}: {yuc.sum():,})")
val

Hogares ENIGH usados: 7,657 (Hidalgo: 1,121)


,ENIGH real (Hidalgo urbano),Imputado (Hidalgo urbano),dif pp
A/B,8.4,8.2,-0.2
C+,13.7,13.6,-0.1
C,15.1,16.6,1.5
C-,19.9,18.6,-1.3
D+,17.3,17.1,-0.2
D,21.1,21.5,0.4
E,4.4,4.4,0.0


## 3. Hogares de la muestra censal (estado) → variables de salida y probabilidad NSE

In [5]:
viv = pd.read_csv(D_MICRO / f"Viviendas{C.ENT}.CSV", dtype=str)
per = pd.read_csv(D_MICRO / f"Personas{C.ENT}.CSV", dtype=str,
                  usecols=["ID_VIV", "EDAD", "PARENTESCO", "NIVACAD", "ESCOLARI", "CONACT", "ESCOACUM"])

viv = viv[viv.NUMPERS.notna() & viv.REFRIGERADOR.notna()].copy()   # viviendas con informacion
for c in ["FACTOR", "NUMPERS", "CUADORM", "TOTCUART", "JEFE_EDAD"]:
    viv[c] = pd.to_numeric(viv[c], errors="coerce")

per["EDAD"] = pd.to_numeric(per.EDAD, errors="coerce").replace(999, np.nan)
per["ESCOACUM"] = pd.to_numeric(per.ESCOACUM, errors="coerce").replace(99, np.nan)
ocupado = per.CONACT.isin(["10", "13", "14", "15", "16", "17", "18", "19", "20"])
e = per.EDAD
nivac = pd.to_numeric(per.NIVACAD, errors="coerce")
per = per.assign(
    n0_5=e.between(0, 5), n6_11=e.between(6, 11), n12_17=e.between(12, 17),
    n18_24=e.between(18, 24), n60=e >= 60, n65=e >= 65,
    ocup12=ocupado & (e >= 12), ocup14=ocupado & (e >= 14),
    p18pb=(e >= 18) & nivac.isin([4, 5, 7, 8, 9, 10, 11, 12, 13, 14]),
    esc15=np.where(e >= 15, per.ESCOACUM, 0),
    edad_menor=np.where(e < 18, e, np.nan),
)
agg = per.groupby("ID_VIV").agg(
    n0_5=("n0_5", "sum"), n6_11=("n6_11", "sum"), n12_17=("n12_17", "sum"), n18_24=("n18_24", "sum"),
    n60=("n60", "sum"), n65=("n65", "sum"), ocup12=("ocup12", "sum"), ocup14=("ocup14", "sum"),
    p18pb=("p18pb", "sum"), esc15=("esc15", "sum"), edad_menor=("edad_menor", "min"))
jefe = per[per.PARENTESCO.str.startswith("1")].drop_duplicates("ID_VIV").set_index("ID_VIV")
agg["pts_edu"] = nse.edu_censo(jefe.NIVACAD, jefe.ESCOLARI).reindex(agg.index)

h = viv.set_index("ID_VIV").join(agg, how="inner")
h["pts_edu"] = h.pts_edu.fillna(h.groupby("MUN").pts_edu.transform("median"))
h["JEFE_EDAD"] = h.JEFE_EDAD.where(h.JEFE_EDAD < 130)
h["JEFE_EDAD"] = h.JEFE_EDAD.fillna(h.JEFE_EDAD.median())

# variables en la misma definicion que el modelo ENIGH
h = h.assign(
    dorm=h.CUADORM.where(h.CUADORM < 99, 1).clip(0, 4), cuartos=h.TOTCUART.where(h.TOTCUART < 99, 2).clip(0, 8),
    internet=(h.INTERNET == "7").astype(int), ocup=h.ocup14, ocup_model=h.ocup12,
    integ=h.NUMPERS.clip(0, 8), pc=(h.COMPUTADORA == "1").astype(int),
    lavadora=(h.LAVADORA == "3").astype(int), micro=(h.HORNO == "5").astype(int),
    banos1=((h.REGADERA == "7") & (h.SERSAN == "1")).astype(int),
    auto1=(h.AUTOPROP == "7").astype(int),
)
_req = [c for c in ["pts_edu", "dorm", "cuartos", "internet", "ocup_model", "integ", "pc", "lavadora", "micro", "banos1", "auto1"] if c in h]
_falt = h[_req].isna().any(axis=1)
if _falt.any():                                       # p. ej. Tlaxcala: viviendas de la muestra sin cuartos o dormitorios
    print(f"Hogares de la muestra sin dato para el NSE: {int(_falt.sum()):,} ({_falt.mean():.2%}); se excluyen")
    h = h[~_falt].copy()
P_NSE = nse.distribucion_nse(h, modelos)

# ---- matriz de categorias de salida (hogares x categorias, valores 0-1) ----
cat = pd.DataFrame(index=h.index)
n = h.NUMPERS
cat["2 o Menos Personas"], cat["3 personas"], cat["4 personas"] = n <= 2, n == 3, n == 4
cat["5 personas"], cat["6+ personas"] = n == 5, n >= 6
m = h.edad_menor
cat["Niño < 6"], cat["Niño 6 - 11"], cat["Niño 12+"], cat["Sin Niños"] = m < 6, m.between(6, 11), m >= 12, m.isna()
a = h.JEFE_EDAD
cat["Edad menos de 35"], cat["Edad 35 - 44"], cat["Edad 45 - 54"] = a < 35, a.between(35, 44), a.between(45, 54)
cat["Edad 55 - 64"], cat["Edad 65+"] = a.between(55, 64), a >= 65
for niv in ["A/B", "C+", "C", "C-", "D+"]:
    cat[niv] = P_NSE[niv]
cat["D/E"] = P_NSE["D"] + P_NSE["E"]
cat = cat.astype(float)[C.CATEGORIAS]
print(f"Hogares en la muestra ({C.NOM_ENT}): {len(h):,}  |  representan {h.FACTOR.sum():,.0f} hogares")

# distribucion de referencia estatal (muestra expandida)
REF_ESTADO = cat.mul(h.FACTOR, axis=0).sum() / h.FACTOR.sum() * 100
REF_ESTADO.round(1).to_frame(f"% {C.NOM_ENT}")

Hogares en la muestra (Hidalgo): 137,698  |  representan 850,674 hogares


,% Hidalgo
2 o Menos Personas,30.2
3 personas,20.1
4 personas,22.6
5 personas,14.3
6+ personas,12.8
Niño < 6,27.5
Niño 6 - 11,17.8
Niño 12+,13.4
Sin Niños,41.3
Edad menos de 35,16.9


## 4. Totales por AGEB del Censo 2020 (restricciones de la microsimulación)

In [6]:
f_ageb = next(D_AGEB.rglob(f"conjunto_de_datos_ageb_urbana_{C.ENT}_cpv2020.csv"))
try:                                                   # Yucatán viene en utf-8-sig, Jalisco en latin-1
    raw = pd.read_csv(f_ageb, dtype=str, encoding="utf-8-sig")
except UnicodeDecodeError:
    raw = pd.read_csv(f_ageb, dtype=str, encoding="latin-1")
# nombres de localidad desde el Marco Geoestadistico (en el CSV AGEB solo dice "Total de la localidad")
loc = gpd.read_file(D_MG / "conjunto_de_datos" / f"{C.ENT}l.shp", columns=["CVEGEO", "NOMGEO"], ignore_geometry=True)
nom_loc = loc.set_index("CVEGEO").NOMGEO
ag = raw[(raw.AGEB != "0000") & (raw.MZA == "000") & raw.MUN.isin(C.ZM_MUNICIPIOS)].copy()
num_cols = ag.columns[8:]
ag[num_cols] = ag[num_cols].apply(pd.to_numeric, errors="coerce")   # '*' y 'N/D' -> NaN
ag = ag.copy()
ag["NOM_LOC"] = (C.ENT + ag.MUN + ag.LOC).map(nom_loc)
ag["CVEGEO"] = C.ENT + ag.MUN + ag.LOC + ag.AGEB
ag = ag[ag.TOTHOG > 0].reset_index(drop=True)

fh = ag.TOTHOG / ag.VIVPARH_CV          # VPH_* estan referidos a viviendas con caracteristicas
fp = ag.POBHOG / ag.POBTOT              # personas -> solo poblacion en hogares
T = pd.DataFrame({
    "hog": ag.TOTHOG,
    "autom": ag.VPH_AUTOM * fh, "inter": ag.VPH_INTER * fh, "pc": ag.VPH_PC * fh,
    "lavad": ag.VPH_LAVAD * fh, "micro": ag.VPH_HMICRO * fh, "stvp": ag.VPH_STVP * fh,
    "spmv": ag.VPH_SPMVPI * fh, "telef": ag.VPH_TELEF * fh, "refri": ag.VPH_REFRI * fh,
    "dor1": ag.VPH_1DOR * fh, "cua1": ag.VPH_1CUART * fh, "cua2": ag.VPH_2CUART * fh,
    "jefe_f": ag.HOGJEF_F,
    "pers": ag.POBHOG, "n0_5": (ag.P_0A2 + ag.P_3A5) * fp, "n6_11": ag.P_6A11 * fp,
    "n12_17": (ag.P_12A14 + ag.P_15A17) * fp, "n18_24": ag.P_18A24 * fp,
    "n60": ag.P_60YMAS * fp, "n65": ag.POB65_MAS * fp, "ocup12": ag.POCUPADA * fp,
    "p18pb": ag.P18YM_PB * fp, "esc15": ag.GRAPROES * ag.P_15YMAS * fp,
})
print(f"AGEB urbanas con hogares en {C.ZM_NOMBRE}: {len(ag):,} | hogares: {ag.TOTHOG.sum():,.0f}")
print("Restricciones faltantes (celdas confidenciales) por variable:")
print(T.isna().sum()[T.isna().sum() > 0].to_dict())

AGEB urbanas con hogares en Estado de Hidalgo (QA): 1,409 | hogares: 525,839
Restricciones faltantes (celdas confidenciales) por variable:
{'autom': 33, 'inter': 52, 'pc': 58, 'lavad': 20, 'micro': 54, 'stvp': 40, 'spmv': 115, 'telef': 99, 'refri': 5, 'dor1': 58, 'cua1': 234, 'cua2': 111, 'jefe_f': 58, 'n0_5': 108, 'n6_11': 37, 'n12_17': 99, 'n18_24': 36, 'n60': 41, 'n65': 55, 'p18pb': 12}


## 5. Microsimulación (IPU) por AGEB

Donantes = hogares de la muestra censal de los 5 municipios de la ZM. Peso inicial = factor de
expansión; los donantes del mismo municipio que la AGEB reciben el doble de peso inicial.

In [7]:
don = h[h.MUN.isin(C.ZM_MUNICIPIOS)]
cat_don = cat.loc[don.index].values
X = np.column_stack([
    np.ones(len(don)), don.auto1, don.internet, don.pc, don.lavadora, don.micro,
    don.SERV_TV_PAGA == "1", don.SERV_PEL_PAGA == "3", don.TELEFONO == "3", don.REFRIGERADOR == "1",
    don.CUADORM == 1, don.TOTCUART == 1, don.TOTCUART == 2, don.JEFE_SEXO == "3",
    don.NUMPERS, don.n0_5, don.n6_11, don.n12_17, don.n18_24, don.n60, don.n65,
    don.ocup12, don.p18pb, don.esc15.fillna(0),
]).astype(float)
assert X.shape[1] == T.shape[1]

mun_ag, mun_don = ag.MUN.to_numpy(dtype=object), don.MUN.to_numpy(dtype=object)
# por lotes de AGEB: cada AGEB se calibra por separado (mismo resultado), y la matriz AGEB × donantes no cabe en memoria en
# estados grandes (Oaxaca: 2,501 AGEB × 523,099 donantes = 9.75 GB). En una ZM como Mérida cabe todo en un solo lote.
f_don = don.FACTOR.to_numpy(float)
LOTE = max(1, int(2e9 // (32 * len(don))))
HH_l, err_l, WX_tot = [], [], np.zeros(X.shape[1])
for i0 in range(0, len(ag), LOTE):
    sl = slice(i0, i0 + LOTE)
    W0 = f_don[None, :] * np.where(mun_ag[sl, None] == mun_don[None, :], 2.0, 1.0)
    W, e_ = ipu(X, T.values[sl].astype(float), W0)
    HH_l.append(W @ cat_don)
    err_l.append(e_)
    WX_tot += (W @ X).sum(axis=0)
    del W, W0
HH, errores = np.vstack(HH_l), np.concatenate(err_l)

# error relativo maximo por AGEB (sobre la restriccion peor ajustada, escala min. 5% de hogares)
print(f"Donantes: {len(don):,} | error máx. de calibración por AGEB — mediana {np.median(errores):.3f}, "
      f"p90 {np.quantile(errores, .9):.3f}")
# ajuste agregado por restriccion (qué tan bien se reproduce el Censo AGEB en conjunto)
ajuste = pd.DataFrame({"Censo AGEB": np.nansum(T.values, axis=0), "Microsim": WX_tot}, index=T.columns)
ajuste["dif %"] = ((ajuste.Microsim / ajuste["Censo AGEB"] - 1) * 100).round(2)
ajuste.round(0)

Donantes: 137,698 | error máx. de calibración por AGEB — mediana 0.109, p90 0.276


,Censo AGEB,Microsim,dif %
hog,525839.0,554136.0,5.0
autom,259287.0,270303.0,4.0
inter,270069.0,278846.0,3.0
pc,211798.0,217747.0,3.0
lavad,364670.0,379731.0,4.0
micro,230213.0,236684.0,3.0
stvp,262983.0,274024.0,4.0
spmv,85286.0,86591.0,2.0
telef,174150.0,181242.0,4.0
refri,462453.0,484301.0,5.0


## 6. Armar la tabla de salida (HHs, %, Índice) + coordenadas

In [8]:
geo = gpd.read_file(D_MG / "conjunto_de_datos" / f"{C.ENT}a.shp")
geo = geo[geo.CVEGEO.isin(ag.CVEGEO)]
pt = geo.set_index("CVEGEO").geometry.representative_point().to_crs(4326)

hh = pd.DataFrame(HH, columns=C.CATEGORIAS)
# cada grupo debe sumar el total de hogares de la AGEB: normalizamos por grupo
for g, cols in C.GRUPOS.items():
    hh[cols] = hh[cols].div(hh[cols].sum(axis=1), axis=0).mul(ag.TOTHOG, axis=0)
pct = pd.DataFrame({c: hh[c] / ag.TOTHOG * 100 for c in C.CATEGORIAS})

REF_ZM = hh.sum() / ag.TOTHOG.sum() * 100
REF = REF_ZM if C.REFERENCIA_INDICE == "zm" else REF_ESTADO
idx = pct.div(REF, axis=1) * 100

ids = pd.DataFrame({
    "CVEGEO": ag.CVEGEO,
    "Latitud": ag.CVEGEO.map(pt.y).round(6), "Longitud": ag.CVEGEO.map(pt.x).round(6),
    "Región Nielsen": C.REGION_NIELSEN, "Estado": C.NOM_ENT, "Municipio": ag.NOM_MUN,
    "Localidad": ag.NOM_LOC, "Zonas Metropolitanas": C.ZM_NOMBRE,
    "Total HHs": ag.TOTHOG, "Población": ag.POBTOT,
})
blocks = {}
for c in C.CATEGORIAS:
    blocks[(c, "HHs")] = hh[c].round(0)
    blocks[(c, "%")] = pct[c].round(1)
    blocks[(c, "Indice")] = idx[c].round(0)
tabla = pd.concat([pd.concat({("", k): v for k, v in ids.items()}, axis=1),
                   pd.DataFrame(blocks)], axis=1)
tabla.columns = pd.MultiIndex.from_tuples(tabla.columns)
tabla.head()

2 o Menos Personas              3 personas               \
          CVEGEO    Latitud   Longitud   Región Nielsen   Estado     Municipio     Localidad    Zonas Metropolitanas Total HHs Población                HHs     % Indice        HHs     % Indice   
0  1300100010043  20.145101 -98.440394  Región Sur (QA)  Hidalgo       Acatlán       Acatlán  Estado de Hidalgo (QA)     130.0       439               43.0  33.2  113.0       23.0  17.8   84.0   
1  1300200010125  20.155842 -98.205302  Región Sur (QA)  Hidalgo  Acaxochitlán  Acaxochitlán  Estado de Hidalgo (QA)     429.0      1753              149.0  34.7  118.0       76.0  17.7   83.0   
2  130020001013A  20.158768 -98.199944  Región Sur (QA)  Hidalgo  Acaxochitlán  Acaxochitlán  Estado de Hidalgo (QA)     557.0      2108              209.0  37.5  127.0      102.0  18.3   86.0   
3  130020001020A  20.154427 -98.201374  Región Sur (QA)  Hidalgo  Acaxochitlán  Acaxochitlán  Estado de Hidalgo (QA)      12.0        47                5.0  40.3  137.0        2.0  17.8   84.0   
4  1300200070214  20.156219 -98.169919  Región Sur (QA)  Hidalgo  Acaxochitlán     Los Reyes  Estado de Hidalgo (QA)     465.0      2192              137.0  29.5  100.0       73.0  15.6   74.0   

  4 personas              5 personas  ... Edad 65+          A/B                C+                  C                 C-                 D+                 D/E               
         HHs     % Indice        HHs  ...        % Indice   HHs    % Indice   HHs     % Indice   HHs     % Indice   HHs     % Indice   HHs     % Indice    HHs     % Indice  
0       28.0  21.9   93.0       18.0  ...     27.7  156.0  10.0  7.5   93.0  15.0  11.2   94.0  18.0  13.5   95.0  18.0  13.5   90.0  15.0  11.7   85.0   55.0  42.5  116.0  
1       91.0  21.2   91.0       59.0  ...     13.8   78.0  22.0  5.1   64.0  34.0   8.0   67.0  44.0  10.3   72.0  48.0  11.2   74.0  51.0  11.8   85.0  230.0  53.6  146.0  
2      110.0  19.7   84.0       72.0  ...     19.5  110.0  19.0  3.4   42.0  31.0   5.6   47.0  43.0   7.8   55.0  54.0   9.7   64.0  64.0  11.5   83.0  346.0  62.1  169.0  
3        2.0  19.1   82.0        1.0  ...     25.2  142.0   1.0  6.2   76.0   1.0   9.3   78.0   1.0  11.8   83.0   1.0  12.4   82.0   2.0  12.9   93.0    6.0  47.4  129.0  
4       89.0  19.0   81.0       79.0  ...     14.4   81.0   0.0  0.0    1.0   1.0   0.3    2.0   5.0   1.2    8.0  16.0   3.5   23.0  44.0   9.5   69.0  397.0  85.5  233.0  

[5 rows x 70 columns]

## 7. Validaciones rápidas

In [9]:
# (a) ZM agregada vs estimación directa de la muestra (ambas deberían parecerse)
don_ref = cat.loc[don.index].mul(don.FACTOR, axis=0).sum() / don.FACTOR.sum() * 100
chk = pd.DataFrame({"ZM (suma AGEB)": REF_ZM, "ZM (muestra directa)": don_ref, C.NOM_ENT: REF_ESTADO}).round(1)
# (b) comparación NSE con AMAI nacional 2022 (publicado): A/B 7.3, C+ 12.0, C 15.3, C- 16.4, D+ 14.9, D/E 34.1
chk

,ZM (suma AGEB),ZM (muestra directa),Hidalgo
2 o Menos Personas,29.5,30.2,30.2
3 personas,21.3,20.1,20.1
4 personas,23.4,22.6,22.6
5 personas,13.7,14.3,14.3
6+ personas,12.1,12.8,12.8
Niño < 6,25.5,27.5,27.5
Niño 6 - 11,17.5,17.8,17.8
Niño 12+,13.7,13.4,13.4
Sin Niños,43.2,41.3,41.3
Edad menos de 35,15.8,16.9,16.9


## 8. Guardar resultados

* `outputs/<ciudad>/01_nse_ageb_zm_<ciudad>.xlsx` – formato del entregable (encabezado de 2 niveles).
* `data/processed/<ciudad>/nse_ageb_zm_<ciudad>.parquet` – tabla plana (para el paso 03).
* `data/processed/<ciudad>/ageb_zm_<ciudad>.gpkg` – polígonos AGEB con los atributos (para mapas).
* `data/processed/<ciudad>/manzanas_zm_<ciudad>.parquet` – centroides de manzana con su participación de hogares
  dentro de la AGEB (para medir áreas de influencia de tiendas en el paso 03).

In [10]:
out_xlsx = C.OUT / f"01_nse_ageb_{C.SLUG}.xlsx"
with pd.ExcelWriter(out_xlsx, engine="openpyxl") as xw:
    tabla.to_excel(xw, sheet_name="AGEB")
    pd.DataFrame({"Referencia %": REF.round(2)}).to_excel(xw, sheet_name="Referencia indice")
    val.to_excel(xw, sheet_name="Validacion ENIGH")
    chk.to_excel(xw, sheet_name="Validacion ZM")
    ajuste.to_excel(xw, sheet_name="Ajuste microsim")

plana = pd.concat([ids, hh.add_prefix("HHs_")], axis=1)
plana.to_parquet(C.PROC / f"nse_ageb_{C.SLUG}.parquet", index=False)
REF_ESTADO.to_frame("pct").to_parquet(C.PROC / "ref_estado.parquet")
geo.merge(plana, on="CVEGEO").to_file(C.PROC / f"ageb_{C.SLUG}.gpkg", driver="GPKG")

# ---- manzanas: participacion de hogares dentro de su AGEB ----
mz = raw[(raw.MZA != "000") & raw.MUN.isin(C.ZM_MUNICIPIOS)][["MUN", "LOC", "AGEB", "MZA", "TOTHOG", "VIVPAR_HAB"]].copy()
mz["CVEGEO_AGEB"] = C.ENT + mz.MUN + mz.LOC + mz.AGEB
mz["CVEGEO"] = mz.CVEGEO_AGEB + mz.MZA
mz["hog"] = pd.to_numeric(mz.TOTHOG, errors="coerce")
mz = mz[mz.CVEGEO_AGEB.isin(ag.CVEGEO)]
tot = ag.set_index("CVEGEO").TOTHOG
conocido = mz.groupby("CVEGEO_AGEB").hog.sum()
# manzanas confidenciales ('*'): reparten a partes iguales el remanente de hogares de su AGEB
masc = mz.hog.isna() & (pd.to_numeric(mz.VIVPAR_HAB, errors="coerce").fillna(1) > 0)
n_masc = masc.groupby(mz.CVEGEO_AGEB).sum()
resto = (tot - conocido.reindex(tot.index).fillna(0)).clip(lower=0)
mz.loc[masc, "hog"] = (resto / n_masc.reindex(tot.index).replace(0, np.nan)).reindex(mz.loc[masc, "CVEGEO_AGEB"]).values
mz["hog"] = mz.hog.fillna(0)
mz["share"] = mz.hog / mz.groupby("CVEGEO_AGEB").hog.transform("sum")
gmz = gpd.read_file(D_MG / "conjunto_de_datos" / f"{C.ENT}m.shp", columns=["CVEGEO"])
gmz = gmz[gmz.CVEGEO.isin(mz.CVEGEO)]
cm = gmz.set_index("CVEGEO").geometry.representative_point().to_crs(4326)
mz["lat"], mz["lon"] = mz.CVEGEO.map(cm.y), mz.CVEGEO.map(cm.x)
sin_poligono = mz.lat.isna() & (mz.hog > 0)
print(f"Manzanas del Censo sin polígono en el Marco Geoestadístico {C.MG_VERSION}: {sin_poligono.sum():,} "
      f"({mz.hog[sin_poligono].sum():,.0f} hogares, {mz.hog[sin_poligono].sum() / mz.hog.sum():.2%}) → en los radios del 03 no entran; "
      "en el 04 se reparten sobre su AGEB")
# Polígonos de manzana para repartir sus hogares por área (notebook 04): los hogares ocupan toda la manzana, no un punto.
# Las manzanas del Censo sin polígono en el Marco vigente se reparten sobre el polígono de su AGEB (quedan dentro, no se pierden).
mzg = mz.dropna(subset=["share"]).merge(gmz[["CVEGEO", "geometry"]], on="CVEGEO", how="left")
falta = mzg.geometry.isna()
mzg.loc[falta, "geometry"] = mzg.loc[falta, "CVEGEO_AGEB"].map(geo.set_index("CVEGEO").geometry).values
mzg["geom_ageb"] = falta
mzg = mzg[mzg.geometry.notna()]
gpd.GeoDataFrame(mzg[["CVEGEO", "CVEGEO_AGEB", "hog", "share", "geom_ageb", "geometry"]], geometry="geometry", crs=gmz.crs).to_file(
    C.PROC / f"manzanas_{C.SLUG}.gpkg", driver="GPKG")
print(f"Polígonos de manzana guardados: {len(mzg):,} ({falta.sum():,} usan el polígono de su AGEB)")
mz = mz.dropna(subset=["lat", "share"])
mz[["CVEGEO", "CVEGEO_AGEB", "hog", "share", "lat", "lon"]].to_parquet(C.PROC / f"manzanas_{C.SLUG}.parquet", index=False)
print("Guardado:", out_xlsx, f"| manzanas: {len(mz):,}")

Manzanas del Censo sin polígono en el Marco Geoestadístico 2025eic: 370 (3,776 hogares, 0.72%) → en los radios del 03 no entran; en el 04 se reparten sobre su AGEB


Polígonos de manzana guardados: 40,223 (502 usan el polígono de su AGEB)
Guardado: C:\Users\KIN\Desktop\Kin\mx-retail-geodemographics\outputs\qasur_13\01_nse_ageb_zm_qasur_13.xlsx | manzanas: 39,734


## 8. Localidades rurales de la ZM (fuera de las AGEB urbanas)

El Censo por AGEB y manzana solo cubre localidades **urbanas**. Los hogares de las localidades rurales (comisarías,
rancherías) también son demanda de las tiendas cercanas, así que deben quedar **dentro** de las áreas de 300 m:
* **Hogares:** ITER 2020 (`TOTHOG` por localidad; si es confidencial `*`, se usa `VIVPAR_HAB`, y si también, 1).
* **Dónde:** manzanas rurales del Marco Geoestadístico vigente de esa localidad (reparto por área); si no tiene,
  el polígono de la localidad; si solo es un punto, un círculo de 100 m.
* **Perfil:** hogares de la muestra censal en localidades de < 2,500 habitantes (`TAMLOC` = 1) del mismo municipio
  (con < 30 hogares en la muestra, los de toda la ZM). Mismo NSE AMAI 2024 que las AGEB.

In [11]:
f_iter = next(D_ITER.rglob("conjunto_de_datos_iter_*.csv"))
try:
    it = pd.read_csv(f_iter, dtype=str, encoding="utf-8-sig")
except UnicodeDecodeError:
    it = pd.read_csv(f_iter, dtype=str, encoding="latin-1")
it = it[it.MUN.isin(C.ZM_MUNICIPIOS) & ~it.LOC.isin(["0000", "9998", "9999"])].copy()
it["CVEGEO"] = C.ENT + it.MUN + it.LOC
urbanas = set(ag.CVEGEO.str[:9])
it = it[~it.CVEGEO.isin(urbanas)]
it["hog"] = pd.to_numeric(it.TOTHOG, errors="coerce").fillna(pd.to_numeric(it.VIVPAR_HAB, errors="coerce")).fillna(1.0)
it = it[it.hog > 0]

# perfil rural por municipio (muestra censal expandida)
rural = h.TAMLOC.astype(str).eq("1") & h.MUN.isin(C.ZM_MUNICIPIOS)
prof_zm = cat[rural].mul(h.FACTOR[rural], axis=0).sum() / h.FACTOR[rural].sum()
def perfil_rural(mun):
    r = rural & h.MUN.eq(mun)
    return cat[r].mul(h.FACTOR[r], axis=0).sum() / h.FACTOR[r].sum() if r.sum() >= 30 else prof_zm
PERFIL_RURAL = pd.DataFrame({m: perfil_rural(m) for m in it.MUN.unique()}).T

# geometría: manzanas rurales → polígono de localidad → punto con 100 m
mzr = gpd.read_file(D_MG / "conjunto_de_datos" / f"{C.ENT}m.shp", columns=["CVEGEO", "AMBITO"])
mzr = mzr[mzr.CVEGEO.str[:9].isin(it.CVEGEO)].assign(LOC9=lambda x: x.CVEGEO.str[:9])
lpol = gpd.read_file(D_MG / "conjunto_de_datos" / f"{C.ENT}l.shp", columns=["CVEGEO"]).to_crs(mzr.crs)
lpt = gpd.read_file(D_MG / "conjunto_de_datos" / f"{C.ENT}lpr.shp", columns=["CVEGEO"]).to_crs(mzr.crs)
partes = [mzr[["LOC9", "geometry"]].rename(columns={"LOC9": "CVEGEO_LOC"}).assign(fuente_geom="manzanas rurales")]
sin_mz = set(it.CVEGEO) - set(mzr.LOC9)
p_l = lpol[lpol.CVEGEO.isin(sin_mz)]
partes.append(p_l.rename(columns={"CVEGEO": "CVEGEO_LOC"}).assign(fuente_geom="polígono de localidad"))
sin_pol = sin_mz - set(p_l.CVEGEO)
p_p = lpt[lpt.CVEGEO.isin(sin_pol)].drop_duplicates("CVEGEO")
p_p = p_p.assign(geometry=p_p.geometry.buffer(100))
partes.append(p_p.rename(columns={"CVEGEO": "CVEGEO_LOC"}).assign(fuente_geom="punto (100 m)"))
rur = gpd.GeoDataFrame(pd.concat(partes, ignore_index=True), geometry="geometry", crs=mzr.crs)
rur["w"] = rur.geometry.area / rur.geometry.area.groupby(rur.CVEGEO_LOC).transform("sum")
rur = rur.merge(it[["CVEGEO", "MUN", "NOM_LOC", "hog"]], left_on="CVEGEO_LOC", right_on="CVEGEO").drop(columns="CVEGEO")
rur = rur.assign(hog=rur.hog.to_numpy(float) * rur.w.to_numpy(float))
P = PERFIL_RURAL.reindex(rur.MUN).to_numpy(dtype=float, copy=True)          # pandas 3: to_numpy() es de solo lectura
for grupo, cols in C.GRUPOS.items():                   # cada grupo suma los hogares de la pieza
    idx = [C.CATEGORIAS.index(c) for c in cols]
    P[:, idx] = P[:, idx] / P[:, idx].sum(axis=1, keepdims=True)
rur = rur.assign(**{f"HHs_{c}": rur.hog.to_numpy(float) * P[:, k] for k, c in enumerate(C.CATEGORIAS)})
sin_geom = set(it.CVEGEO) - set(rur.CVEGEO_LOC)
rur.drop(columns="w").to_file(C.PROC / f"rurales_{C.SLUG}.gpkg", driver="GPKG")
print(f"Localidades rurales en la ZM: {len(it):,} con {it.hog.sum():,.0f} hogares "
      f"({it.hog.sum() / (it.hog.sum() + ag.TOTHOG.sum()):.1%} de los hogares de la ZM) | sin geometría: {len(sin_geom)} "
      f"({it[it.CVEGEO.isin(sin_geom)].hog.sum():,.0f} hogares)")
print(rur.groupby("fuente_geom").agg(localidades=("CVEGEO_LOC", "nunique"), hogares=("hog", "sum")).round(0))
print(f"Hogares rurales en la muestra: {rural.sum():,} | municipios con perfil propio: "
      f"{sum((rural & h.MUN.eq(m)).sum() >= 30 for m in it.MUN.unique())} de {it.MUN.nunique()}")
PERFIL_RURAL.mul(100).round(1)

Localidades rurales en la ZM: 4,422 con 331,117 hogares (38.6% de los hogares de la ZM) | sin geometría: 9 (28 hogares)
                  localidades   hogares
fuente_geom                            
manzanas rurales         2256  303667.0
punto (100 m)            2157   27422.0
Hogares rurales en la muestra: 74,162 | municipios con perfil propio: 84 de 84


,2 o Menos Personas,3 personas,4 personas,5 personas,6+ personas,Niño < 6,Niño 6 - 11,Niño 12+,Sin Niños,Edad menos de 35,Edad 35 - 44,Edad 45 - 54,Edad 55 - 64,Edad 65+,A/B,C+,C,C-,D+,D/E
001,28.4,16.6,22.0,17.8,15.3,32.8,18.4,12.9,35.8,17.9,21.8,21.2,17.6,21.5,1.8,4.5,8.8,13.5,16.6,54.7
002,20.7,16.4,19.0,14.9,29.0,43.7,18.2,11.2,26.9,23.1,23.9,19.2,15.3,18.4,0.0,0.6,3.1,5.6,10.6,80.0
003,32.4,16.8,23.4,15.1,12.3,26.7,16.9,14.2,42.2,18.2,19.0,26.9,14.5,21.5,3.4,7.1,13.0,14.5,17.4,44.7
004,31.0,18.6,22.5,14.5,13.5,30.6,19.8,12.7,36.9,17.9,23.2,20.3,16.5,22.1,0.7,1.8,5.0,9.6,13.2,69.7
005,32.1,15.8,24.6,12.9,14.6,28.7,19.1,9.7,42.5,17.1,17.5,20.6,19.7,25.2,1.2,3.6,10.0,14.4,19.1,51.7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
080,31.2,19.9,19.7,13.6,15.6,29.3,16.3,13.0,41.3,17.7,18.7,19.0,17.5,27.2,0.5,0.6,1.4,2.0,2.6,92.9
081,31.1,21.2,25.7,10.6,11.4,28.3,16.2,14.3,41.1,17.2,18.7,20.6,15.9,27.6,0.4,2.0,4.0,8.0,10.7,75.0
082,28.0,25.4,25.4,10.6,10.6,29.6,21.2,19.0,30.2,27.5,31.7,21.2,6.3,13.2,0.3,1.8,8.7,15.8,24.7,48.7
083,27.5,17.8,25.1,15.5,14.2,29.0,20.2,11.3,39.5,16.7,23.7,20.0,18.2,21.4,3.3,6.1,10.4,15.3,15.9,49.0


## 9. Actualización a 2025 con la Encuesta Intercensal 2025 (por municipio)

La EIC 2025 (INEGI, publicada 2026-09-22) es la cifra oficial más reciente, pero **solo llega a municipio** (no hay AGEB
ni manzana), así que la estructura por AGEB y manzana sigue siendo la del Censo 2020 y se escala por municipio.
* **Se usa el crecimiento de población** (`POBTOT` EIC 2025 / Censo 2020) como factor de los hogares.
* **No se usan los hogares de la EIC directamente:** en la EIC hogar = vivienda (`TOTHOG = VIVPARHAB`) y su cociente
  contra el Censo implica que el tamaño del hogar cayó ~10% en 5 años (p. ej. Mérida 3.3 → 2.9), lo que no es
  demográficamente creíble; es un cambio de definición. Con la población el factor es conservador.
* El notebook 04 aplica el factor a los hogares de cada manzana y localidad rural (demanda de las tiendas en 2025).
  Las tablas por AGEB de este notebook se quedan en Censo 2020 (la base oficial).

In [12]:
f_eic = next(D_EIC.rglob("conjunto_datos_eic2025_105.csv"))
eic = pd.read_csv(f_eic, dtype=str, encoding="latin-1")
eic = eic[(eic.CVE_ENT == C.ENT) & eic.CVE_MUN.isin(C.ZM_MUNICIPIOS) & (eic.CVE_LOC == "0000")]
val = eic[eic.ESTIMADOR == "Valor"].set_index("CVE_MUN")
cv = eic[eic.ESTIMADOR.str.startswith("Coeficiente")].set_index("CVE_MUN")
COLS_ITER = ["MUN", "LOC", "POBTOT", "TOTHOG", "VIVPAR_HAB"]
try:
    base = pd.read_csv(f_iter, dtype=str, encoding="utf-8-sig", usecols=COLS_ITER)
except UnicodeDecodeError:
    base = pd.read_csv(f_iter, dtype=str, encoding="latin-1", usecols=COLS_ITER)
base = base[(base.LOC == "0000") & base.MUN.isin(C.ZM_MUNICIPIOS)].set_index("MUN").apply(pd.to_numeric)
crec = pd.DataFrame({
    "Municipio": pd.Series(C.ZM_MUNICIPIOS),
    "Población 2020 (Censo)": base.POBTOT, "Población 2025 (EIC)": pd.to_numeric(val.POBTOT),
    "CV población EIC %": pd.to_numeric(cv.POBTOT),
    "Hogares 2020 (Censo)": base.TOTHOG, "Hogares 2025 (EIC, hogar = vivienda)": pd.to_numeric(val.TOTHOG)})
crec["Personas por hogar 2020"] = crec["Población 2020 (Censo)"] / crec["Hogares 2020 (Censo)"]
crec["Personas por hogar 2025 (EIC)"] = crec["Población 2025 (EIC)"] / crec["Hogares 2025 (EIC, hogar = vivienda)"]
crec["factor_hogares"] = crec["Población 2025 (EIC)"] / crec["Población 2020 (Censo)"]
crec.index.name = "MUN"
crec.to_parquet(C.PROC / f"crecimiento_municipal_{C.SLUG}.parquet")
print(f"ZM: población 2020 → 2025: {crec['Población 2020 (Censo)'].sum():,.0f} → {crec['Población 2025 (EIC)'].sum():,.0f} "
      f"(factor {crec['Población 2025 (EIC)'].sum() / crec['Población 2020 (Censo)'].sum():.3f})")
crec.round(3)

ZM: población 2020 → 2025: 3,082,841 → 3,237,527 (factor 1.050)


,Municipio,Población 2020 (Censo),Población 2025 (EIC),CV población EIC %,Hogares 2020 (Censo),"Hogares 2025 (EIC, hogar = vivienda)",Personas por hogar 2020,Personas por hogar 2025 (EIC),factor_hogares
MUN,,,,,,,,,
001,Acatlán,22268,23321,6.47,5869,6797,3.794,3.431,1.047
002,Acaxochitlán,46065,49443,6.60,10442,12145,4.412,4.071,1.073
003,Actopan,61002,59733,5.16,16266,18086,3.750,3.303,0.979
004,Agua Blanca de Iturbide,10313,10251,14.58,2854,3086,3.614,3.322,0.994
005,Ajacuba,18872,19922,5.83,5174,5953,3.647,3.347,1.056
...,...,...,...,...,...,...,...,...,...
080,Yahualica,24674,25400,9.06,6277,6963,3.931,3.648,1.029
081,Zacualtipán de Ángeles,38155,42643,7.45,10594,13652,3.602,3.124,1.118
082,Zapotlán de Juárez,21443,24079,5.93,5849,7169,3.666,3.359,1.123
